# 06 — Integrated Decision Dashboard
**Tying All Three Modules Together**

This notebook is the executive summary layer. It pulls outputs from Modules 1–3 and assembles them into a unified view that answers the three business questions in sequence:

1. **Where should we enter first?** (Module 1)
2. **Who should we extend credit to?** (Module 2)
3. **Who needs a retention intervention today?** (Module 3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT   = Path("..")
TABLES = ROOT / "outputs" / "tables"
FIGS   = ROOT / "outputs" / "figures"
SIM    = ROOT / "data" / "simulated"
MODELS = ROOT / "outputs" / "models"

print("Environment ready.")

## 6.1 Module 1 Summary — Market Entry Recommendation

In [ ]:
scorecard = pd.read_csv(TABLES / "market_entry_scorecard.csv")

score_display = scorecard[["rank", "country", "total_score"]].copy()
score_display["total_score"] = score_display["total_score"].round(3)
score_display.columns = ["Rank", "Market", "Weighted Score"]

fig, ax = plt.subplots(figsize=(7, 3.5))
colors = ["#1565C0", "#2196F3", "#90CAF9", "#BBDEFB"]
bars = ax.barh(score_display["Market"][::-1], score_display["Weighted Score"][::-1], color=colors[::-1])
ax.set_xlabel("Weighted Entry Score")
ax.set_title("Market Entry Ranking — Module 1 Output", fontsize=12)
for bar, score in zip(bars, score_display["Weighted Score"][::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{score:.3f}", va="center", fontsize=10)
plt.tight_layout()
plt.savefig(FIGS / "06_entry_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

print(score_display.to_string(index=False))

## 6.2 Module 2 Summary — Credit Policy Simulator

In [ ]:
# Load model + simulate a new applicant batch
with open(MODELS / "credit_risk_model.pkl", "rb") as f:
    bundle = pickle.load(f)

model    = bundle["model"]
features = bundle["features"]

# Simulate 500 new applicants
rng = np.random.default_rng(99)
n   = 500

applicants = pd.DataFrame({
    "downpayment_pct":   rng.beta(2, 5, n) * 0.35,
    "contract_months":   rng.choice([12,18,24,30,36], n),
    "is_urban":          rng.binomial(1, 0.45, n),
    "gender_female_hh":  rng.binomial(1, 0.38, n),
    "age_head":          rng.integers(20, 65, n),
    "wealth_index":      rng.normal(0, 1, n),
    "mobile_money_freq": rng.choice([0,1,2,3], n),
    "prior_credit":      rng.binomial(1, 0.40, n),
    "household_size":    rng.integers(2, 9, n),
})

applicants["default_probability"] = model.predict_proba(applicants[features])[:, 1]
applicants["decision"] = applicants["default_probability"].apply(
    lambda p: "Approve" if p < 0.30 else ("Review" if p < 0.50 else "Decline")
)

decision_summary = applicants["decision"].value_counts().rename_axis("Decision").reset_index(name="Count")
decision_summary["Pct"] = (decision_summary["Count"] / n * 100).round(1)
print(decision_summary.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 3.5))
colors_d = {"Approve": "#4CAF50", "Review": "#FF9800", "Decline": "#EF5350"}
ax.bar(decision_summary["Decision"],
       decision_summary["Count"],
       color=[colors_d[d] for d in decision_summary["Decision"]])
ax.set_ylabel("Number of Applicants")
ax.set_title("Credit Policy Simulation — Module 2 Output", fontsize=12)
plt.tight_layout()
plt.savefig(FIGS / "06_credit_decisions.png", dpi=150, bbox_inches="tight")
plt.show()

## 6.3 Module 3 Summary — Retention Priority Queue

In [ ]:
churn_df = pd.read_csv(SIM / "churn_customer_ledger.csv")

if "churn_risk_score" in churn_df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    churn_df["churn_risk_score"].hist(bins=40, ax=ax, color="#EF5350", alpha=0.75, edgecolor="white")
    ax.axvline(0.50, color="black", linestyle="--", label="Intervention threshold")
    ax.set_xlabel("30-Day Churn Risk Score")
    ax.set_ylabel("Number of Customers")
    ax.set_title("Customer Risk Score Distribution — Module 3 Output", fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGS / "06_churn_risk_dist.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    high_risk_n = (churn_df["churn_risk_score"] > 0.50).sum()
    print(f"Customers flagged for retention intervention: {high_risk_n:,} ({high_risk_n/len(churn_df):.1%} of portfolio)")
else:
    print("Run Notebook 05 first to generate churn risk scores.")

## 6.4 One-Page Board Summary

In [ ]:
fig = plt.figure(figsize=(14, 9))
fig.suptitle("PayGo Cookstove — East Africa Market Entry Analysis", fontsize=16, fontweight="bold", y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# Panel 1 — Radar chart
ax1 = fig.add_subplot(gs[0, 0])
radar_img = plt.imread(FIGS / "02_country_radar.png")
ax1.imshow(radar_img)
ax1.axis("off")
ax1.set_title("Module 1: Country Profiles", fontsize=10, pad=4)

# Panel 2 — Scorecard heatmap
ax2 = fig.add_subplot(gs[0, 1])
hm_img = plt.imread(FIGS / "03_scorecard_heatmap.png")
ax2.imshow(hm_img)
ax2.axis("off")
ax2.set_title("Module 1: Entry Scorecard", fontsize=10, pad=4)

# Panel 3 — Default by DP
ax3 = fig.add_subplot(gs[0, 2])
dp_img = plt.imread(FIGS / "04_default_by_dp.png")
ax3.imshow(dp_img)
ax3.axis("off")
ax3.set_title("Module 2: Default by Downpayment", fontsize=10, pad=4)

# Panel 4 — ROC curves
ax4 = fig.add_subplot(gs[1, 0])
roc_img = plt.imread(FIGS / "04_model_curves.png")
ax4.imshow(roc_img)
ax4.axis("off")
ax4.set_title("Module 2: Model Performance", fontsize=10, pad=4)

# Panel 5 — KM survival
ax5 = fig.add_subplot(gs[1, 1])
km_img = plt.imread(FIGS / "05_km_survival.png")
ax5.imshow(km_img)
ax5.axis("off")
ax5.set_title("Module 3: Survival Curves", fontsize=10, pad=4)

# Panel 6 — Feature importance
ax6 = fig.add_subplot(gs[1, 2])
fi_img = plt.imread(FIGS / "04_feature_importance.png")
ax6.imshow(fi_img)
ax6.axis("off")
ax6.set_title("Module 2: Feature Importance", fontsize=10, pad=4)

plt.savefig(FIGS / "06_integrated_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()

print("Dashboard saved to outputs/figures/06_integrated_dashboard.png")

---

## Executive Summary

| Question | Finding |
|---|---|
| **Where to enter first?** | Kenya ranks #1 on overall readiness (mobile money, financial inclusion, LG MAI). Tanzania is the natural second market. Ethiopia shows the largest raw opportunity but requires a 3–5 year infrastructure thesis. |
| **Who to extend credit to?** | Minimum 10% downpayment requirement alone significantly improves portfolio quality. The XGBoost model identifies high-risk applicants at point of onboarding with AUC ~0.82. |
| **Who to call today?** | The churn model flags ~X% of active customers as high-risk within a 30-day window, enabling proactive field agent intervention before lockout. |